In [1]:
#| default_exp build

In [2]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
import subprocess
from kavacha.spec import App

Building the bundle: the right interpreter, a pinned environment, and a stamp saying which commit.

py2app builds on macOS and py2exe builds on Windows, each on its own operating system. Describing
a build and asking what it would do works anywhere. `check`, the stamp functions and this page all
run on Linux, where no bundle can be built at all, and nothing here builds one.

In [3]:
#| export
from __future__ import annotations
import json, os, shutil, subprocess, sys, time
from fastcore.all import Path

from kavacha.probe import framework_python, is_framework, py_version, running_from
from kavacha.bundle import finish

In [4]:
#| export
#: Written into the bundle at build; read back to tell a stale install from a current one.
STAMP = 'build.json'
#: The marker that says a build already re-execed, so a bad environment reports rather than forks.
REEXEC = 'KAVACHA_BUILD_VENV'

def run(*args, **kw):
    print('·', ' '.join(str(a) for a in args))
    return subprocess.run([str(a) for a in args], check=True, **kw)

`STAMP` names the file a build writes inside the bundle, and `read_stamp` reads back. A bundle
without one was not built by this code.

`REEXEC` is set in the environment of a re-execed build. It is how the second attempt knows it is
the second: an interpreter that is still the wrong one reports rather than forking again.

Every command a build runs goes through `run`. It prints the command before running it, so a build
log names what ran. A non-zero exit raises `CalledProcessError`.

In [5]:
run('echo', 'built')

· echo built


CompletedProcess(args=['echo', 'built'], returncode=0)

In [6]:
#| hide
test_fail(lambda: run('false'), contains='non-zero')

· false


In [7]:
#| export
def lock_requirements(root, extras=(), out=None):
    """`uv.lock` as a pinned requirements file, or None where uv cannot answer.

    pip resolves `>=` against whatever it finds and keeps whatever it already has, so a build
    environment goes on shipping the versions it was first made with while the repository moves.
    The lock is the tested set, and the bundle should carry that one.
    """
    root = Path(root)
    if not (root/'uv.lock').exists(): return None
    args = ['uv', 'export', '--frozen', '--no-dev', '--no-hashes', '--no-emit-project',
            '--format', 'requirements-txt']
    for e in extras: args += ['--extra', e]
    try: got = subprocess.run(args, cwd=str(root), capture_output=True, text=True, timeout=300)
    except (OSError, subprocess.SubprocessError): return None
    if got.returncode or not got.stdout.strip():
        print(f'· uv export failed; resolving from pyproject instead: {got.stderr.strip()[:200]}')
        return None
    out = Path(out or root/'packaging'/'.app-venv'/'app-requirements.txt')
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(got.stdout)
    return out

`lock_requirements` writes `uv.lock` out as a pinned requirements file for the build environment to
install from.

It returns `None` when the tree has no `uv.lock`, when `uv` is not installed, and when `uv export`
fails. None of the three raises. `build_venv` reads `None` as "resolve from `pyproject.toml`
instead", so a project without a lock still builds. It ships whatever pip resolved on the day.

`out` defaults to `packaging/.app-venv/app-requirements.txt` under `root`, and its parent directory
is created. Nothing is written on any of the `None` paths.

In [8]:
tmp = TemporaryDirectory(); root = Path(tmp.name)
lock_requirements(root) is None

True

In [9]:
#| hide
(root/'uv.lock').write_text('not toml')
test_is(lock_requirements(root), None)      # uv rejects the lock, or uv is absent; either way, no pins
assert not (root/'packaging').exists(), 'nothing written when there is nothing to write'

· uv export failed; resolving from pyproject instead: error: No `pyproject.toml` found in current directory or any parent directory


In [10]:
#| export
def build_venv(python, venv, root, extras=(), force=False):
    "A build environment on `python`: a plain `venv`, since `uv venv` links what we are avoiding."
    venv = Path(venv)
    if force: shutil.rmtree(venv, ignore_errors=True)
    exe = venv/'bin'/'python'
    # A build environment left over from another interpreter is not reusable, and reusing one in
    # silence is how an app goes on shipping an old version while the build prints the new one.
    if exe.exists() and py_version(exe) != py_version(python):
        print(f'· rebuilding {venv.name}: it is on {py_version(exe)}, this build wants {py_version(python)}')
        shutil.rmtree(venv, ignore_errors=True)
    if not exe.exists():
        venv.parent.mkdir(parents=True, exist_ok=True)
        run(python, '-m', 'venv', venv)
    run(exe, '-m', 'pip', 'install', '--upgrade', '--quiet', 'pip', 'setuptools', 'wheel')
    spec = f'{root}[{",".join(extras)}]' if extras else str(root)
    if (req := lock_requirements(root, extras, venv/'app-requirements.txt')) is not None:
        run(exe, '-m', 'pip', 'install', '--quiet', '--upgrade', '-r', req)
        run(exe, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', spec)
    else: run(exe, '-m', 'pip', 'install', '--quiet', '-e', spec)
    return exe

`build_venv` makes the environment the freezer runs in, on `python` rather than on this
interpreter. It is a plain `venv`. `uv venv` links back to the interpreter it was made from, which
is the thing a framework build is chosen to avoid.

An environment already there on a different `(major, minor)` than `python` is deleted and remade.
Reusing one is how an app goes on shipping the standard library of an interpreter nobody meant to
build against.

With a lock the environment installs the pinned set first, then the project with `--no-deps`, so
the lock decides every version. Without one pip resolves from `pyproject.toml`.

Nothing on this page calls `build_venv`. It creates a virtual environment and installs into it.

In [11]:
#| export
def git_stamp(root, version=''):
    "The commit the tree is on and whether it is dirty, or `None` outside a checkout."
    def git(*a):
        r = subprocess.run(['git', *a], cwd=str(root), capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if not r.returncode else None
    try: sha = git('rev-parse', 'HEAD')
    except (OSError, subprocess.SubprocessError): return None
    if not sha: return None
    return {'commit': sha, 'dirty': bool(git('status', '--porcelain', '--untracked-files=no')),
            'version': version, 'built': time.strftime('%Y-%m-%dT%H:%M:%S')}

def stamp_path(bundle):
    "Where the build stamp lives inside `bundle`, on either platform."
    b = Path(bundle)
    return (b/'Contents'/'Resources'/STAMP) if b.suffix == '.app' else (b/STAMP)

def write_stamp(bundle, root, version=''):
    """Record which commit the bundle was built from.

    A bundle is derived from the tree and nothing else compares them, so an app built before a fix
    installs over one built after it and reports the version it always did.
    """
    if (st := git_stamp(root, version)) is None:
        st = {'commit': '', 'dirty': False, 'version': version,
              'built': time.strftime('%Y-%m-%dT%H:%M:%S')}
    p = stamp_path(bundle)
    if p.parent.is_dir(): p.write_text(json.dumps(st, indent=1) + '\n')
    return st

def read_stamp(bundle):
    "The stamp `bundle` was built with, or None when it has none."
    try: return json.loads(stamp_path(bundle).read_text())
    except (OSError, ValueError): return None

A bundle is a copy of a tree with nothing pointing back at it. The stamp is the pointer back.

`stamp_path` puts it inside `Contents/Resources` for a `.app` and beside the executable for
anything else. The `.app` suffix is the whole test, so a path names a place before either exists.

In [12]:
stamp_path('/dist/Demo.app'), stamp_path('/dist/Demo')

(Path('/dist/Demo.app/Contents/Resources/build.json'),
 Path('/dist/Demo/build.json'))

`git_stamp` reports the commit, whether the tree had uncommitted changes, the version and the time.
It returns `None` outside a checkout and when `git` is not installed.

`write_stamp` writes that into the bundle and returns it. Outside a checkout it still writes one,
with `commit` empty: a build from a tarball has no commit to name, and that is a fact to record
rather than a failure. It writes no file when the directory it would write into is missing, which
is what a build that produced no bundle leaves behind, and returns the stamp anyway.

`read_stamp` returns `None` for a bundle with no stamp and for a stamp that is not readable JSON.

In [13]:
#| hide
def mkrepo(d):
    "A checkout with one commit in it, which is what a stamp reads."
    def git(*a): subprocess.run(['git', *a], cwd=d, capture_output=True, check=True)
    git('init', '-q'); git('config', 'user.email', 'a@b.c'); git('config', 'user.name', 'T')
    (Path(d)/'x.txt').write_text('one'); git('add', '-A'); git('commit', '-qm', 'first')
    return Path(d)
t2 = TemporaryDirectory(); repo = mkrepo(t2.name)
(repo/'dist'/'Demo.app'/'Contents'/'Resources').mkdir(parents=True)

In [14]:
st = write_stamp(repo/'dist'/'Demo.app', repo, version='2.0.1')
st['commit'][:12], st['dirty'], st['version']

('3b2bbb3a952a', False, '2.0.1')

In [15]:
read_stamp(repo/'dist'/'Demo.app') == st

True

A tree with changes nobody committed stamps `dirty`, and the build prints that alongside the
commit. An app that misbehaves can then be traced to a tree that was never pushed.

In [16]:
(repo/'x.txt').write_text('changed')
git_stamp(repo)['dirty']

True

In [17]:
#| hide
t3 = TemporaryDirectory(); plain = Path(t3.name)
gone = plain/'dist'/'Demo'                                  # the freezer wrote no bundle
test_eq(write_stamp(gone, repo, '2.0.1')['version'], '2.0.1')
assert not gone.exists(), 'no bundle, no stamp file'
test_is(read_stamp(gone), None)
stamp_path(plain).write_text('{ truncated')                 # a half-written stamp is not a stamp
test_is(read_stamp(plain), None)

In [18]:
#| export
def check(spec, root, venv=None):
    "What a build would do here, without doing it. Answers on any platform, Linux included."
    root, venv = Path(root), Path(venv or Path(root)/'packaging'/'.app-venv')
    rows = {'platform': sys.platform, 'interpreter': sys.executable,
            'app': spec.name, 'out': str(spec.out(root))}
    if sys.platform not in ('darwin', 'win32'):
        rows['freezer'] = f'none — {sys.platform} builds nothing; macOS and Windows build on their own OS'
    else: rows['freezer'] = 'py2app' if sys.platform == 'darwin' else 'py2exe'
    if sys.platform == 'darwin':
        rows['framework_build'] = is_framework()
        if not rows['framework_build']:
            rows['would_rebuild'] = str(venv)
            rows['on'] = str(framework_python() or '')
    try:
        import webview  # noqa: F401
        rows['pywebview'] = 'installed'
    except ImportError: rows['pywebview'] = 'MISSING'
    rows['running'] = running_from(spec.out(root))
    return rows

`check` reports what a build would do here, and does none of it.

`freezer` is py2app on macOS, py2exe on Windows, and nothing anywhere else, because each builds
only on its own operating system. `out` is where the bundle would be written. `running` holds the
pids already running out of that path, which is what `build` refuses to overwrite. `pywebview`
reads `MISSING` rather than raising, since the machine planning a build often is not the machine
that runs the app. On macOS `check` also reports whether this interpreter is a framework build, and
which interpreter the build would re-exec into when it is not.

In [19]:
demo = App(name='Demo', entry='demo_app.py', version='2.0.1')
rep = check(demo, repo)
# the interpreter and the output folder are this machine's; the rest is the same anywhere
{**rep, 'interpreter': '/proj/.venv/bin/python3', 'out': '/proj/dist/Demo'}

{'platform': 'linux',
 'interpreter': '/proj/.venv/bin/python3',
 'app': 'Demo',
 'out': '/proj/dist/Demo',
 'freezer': 'none — linux builds nothing; macOS and Windows build on their own OS',
 'pywebview': 'MISSING',
 'running': []}

In [20]:
#| export
def build(spec, root, setup_py, venv=None, alias=False, force=False, rebuild_venv=False,
          identity=None):
    """Build `spec` into `root/dist`, re-execing into a framework Python where macOS needs one.

    `setup_py` is the file that calls `setuptools.setup` with the freezer options; it runs in the
    build environment rather than this one, which is the whole reason for the re-exec.
    """
    root, venv = Path(root), Path(venv or Path(root)/'packaging'/'.app-venv')
    out = spec.out(root)
    if not force and (live := running_from(out)):
        raise SystemExit(
            f'{out} is running as pid {", ".join(map(str, live))}. Quit it first, or pass force.\n'
            'A build replaces the standard library the running app imports from, and it fails from '
            'then on with a zip error that says nothing about the build.')
    if sys.platform == 'darwin' and not is_framework():
        if os.environ.get(REEXEC):
            raise SystemExit(f'{sys.executable} is still not a framework build; run `check` to see why')
        if (found := framework_python()) is None:
            raise SystemExit('py2app needs a framework Python, and this one is a standalone build '
                             'whose stdlib extension modules are not files.\n'
                             'Install one and try again:  brew install python@3.12')
        print(f'framework Python: {found}')
        exe = build_venv(found, venv, root, spec.extras, force=rebuild_venv)
        run(exe, sys.argv[0], *(['--alias'] if alias else []), env={**os.environ, REEXEC: '1'})
        return out
    args = [sys.executable, str(setup_py)] + (['py2app'] if sys.platform == 'darwin' else [])
    if alias and sys.platform == 'darwin': args.append('--alias')
    run(*args, cwd=str(root))
    if out.exists() and sys.platform == 'darwin':
        for name, where in finish(out, spec, identity).items(): print(f'  {name}: {where}')
    st = write_stamp(out, root, spec.version)
    print(f"\nbuilt {out}\n  from {st['commit'][:12]}"
          f"{' with uncommitted changes' if st['dirty'] else ''}")
    return out

`build` runs the freezer, and on macOS it may have to run it under a different interpreter than the
one it was called with.

py2app copies the interpreter's standard library into the bundle, and only a framework build keeps
that library as files it can copy. When this interpreter is not one, `build` finds a framework
Python, makes a build environment on it with `build_venv`, and runs itself again under that
interpreter with `REEXEC` set in the environment. The re-execed run that is still not a framework
build raises `SystemExit`. There is no third attempt.

`setup_py` is run as a subprocess rather than imported. That is what lets it run under the
re-execed interpreter, and it is the whole reason the re-exec works.

A build over a running app raises `SystemExit` first. py2app writes the bundle in place, and
replacing the standard library under a live process breaks every import that process has not made
yet, with a zip error naming nothing to do with a build. `force` builds anyway.

`write_stamp` runs last, so the bundle on disk names the commit it came from.

Nothing on this page calls `build`. It needs a freezer, and `check` is what answers without one.

In [21]:
#| hide
for t in (tmp, t2, t3): t.cleanup()